# 4: Hybrid Search + RRF Fusion + CrossEncoder Re-Ranking
## RAG-Based Medical QA System — PubMedQA
---
**What this notebook does:**
- Implements Semantic Search (Pinecone)
- Implements BM25 Keyword Search
- Fuses both results using Reciprocal Rank Fusion (RRF)
- Re-ranks fused results using CrossEncoder
- Tests the full retrieval pipeline
- Saves the retrieval module for use in the final app


## 4.1 Install & Import

In [ ]:
!pip install sentence-transformers pinecone rank-bm25 tqdm -q
print('Done!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 29.6 MB/s eta 0:00:00
Done!


In [ ]:
import json
import time
import pickle
import numpy as np
from tqdm import tqdm
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from collections import defaultdict

# ============================================================
#  FILL IN YOUR KEY
# ============================================================
PINECONE_API_KEY = 'pcsk_4EURsR_7vhX2ipcBnChK6GbiyWKXFFWTgXjCTezhT1MuZufSHd5ehrHNdJQsRjHuBrvuha'
INDEX_NAME       = 'pubmed-rag'
# ============================================================

print('Imports done!')

Imports done!


## 4.2 Upload & Load BM25 Index

In [ ]:
from google.colab import files
print('Upload bm25_index.pkl')
uploaded = files.upload()

Upload bm25_index.pkl


Saving bm25_index.pkl to bm25_index.pkl


In [ ]:
with open('bm25_index.pkl', 'rb') as f:
    bm25_data = pickle.load(f)

bm25_index        = bm25_data['bm25']
all_chunks        = bm25_data['chunks']
tokenized_corpus  = bm25_data['tokenized_corpus']

print(f'BM25 index loaded over {len(all_chunks)} chunks ✅')

BM25 index loaded over 11605 chunks ✅


## 4.3 Load Models

In [ ]:
# Bi-encoder for query embedding (same model used during indexing)
print('Loading bi-encoder (all-MiniLM-L6-v2)...')
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
print('Bi-encoder loaded ✅')

# CrossEncoder for re-ranking
# ms-marco model is trained specifically for passage re-ranking
print('Loading CrossEncoder (ms-marco-MiniLM-L-6-v2)...')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('CrossEncoder loaded ✅')

Loading bi-encoder (all-MiniLM-L6-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

CrossEncoder loaded ✅


## 4.4 Connect to Pinecone

In [ ]:
pc    = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)

stats = index.describe_index_stats()
print(f'Pinecone connected ✅  |  Vectors: {stats.total_vector_count}')

Pinecone connected ✅  |  Vectors: 11605


## 4.5 Component 1 — Semantic Search
Encodes query → queries Pinecone → returns top-k chunks by cosine similarity.

In [ ]:
def semantic_search(query, bi_encoder, pinecone_index, top_k=20):
    """
    Dense vector search via Pinecone.

    Returns:
        List of dicts: {chunk_id, text, score, doc_id, pubid}
    """
    # Encode query
    query_emb = bi_encoder.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0].tolist()

    # Query Pinecone
    results = pinecone_index.query(
        vector=query_emb,
        top_k=top_k,
        include_metadata=True
    )

    # Format results
    hits = []
    for match in results['matches']:
        hits.append({
            'chunk_id': match['id'],
            'text'    : match['metadata']['text'],
            'score'   : match['score'],
            'doc_id'  : match['metadata'].get('doc_id', ''),
            'pubid'   : match['metadata'].get('pubid', '')
        })

    return hits


# Quick test
test_q = 'Does aspirin reduce cardiovascular risk?'
sem_results = semantic_search(test_q, bi_encoder, index, top_k=5)
print(f'Semantic search test — top 3 results:')
for i, r in enumerate(sem_results[:3]):
    print(f'  [{i+1}] score={r["score"]:.4f} | {r["text"][:100]}...')

Semantic search test — top 3 results:
  [1] score=0.6976 | Since the 1980s, clinical trial evidence has supported aspirin use in the secondary prevention of ca...
  [2] score=0.6712 | . Overall, 547 men (9.5%) were taking aspirin daily, of whom 321 (59%) had documented CVD. Among men...
  [3] score=0.5665 | . 1 year after GAP implementation, adherence to most medications remained high. We found a significa...


## 4.6 Component 2 — BM25 Keyword Search
Classic TF-IDF style keyword matching. Fast and great for medical terminology.

In [ ]:
def bm25_search(query, bm25_index, all_chunks, top_k=20):
    """
    Sparse keyword search using BM25Okapi.

    Returns:
        List of dicts: {chunk_id, text, score, doc_id, pubid}
    """
    # Tokenize query same way as corpus
    tokenized_query = query.lower().split()

    # Get BM25 scores for all chunks
    scores = bm25_index.get_scores(tokenized_query)

    # Get top-k indices
    top_indices = np.argsort(scores)[::-1][:top_k]

    hits = []
    for idx in top_indices:
        if scores[idx] > 0:  # only include chunks with positive score
            hits.append({
                'chunk_id': all_chunks[idx]['chunk_id'],
                'text'    : all_chunks[idx]['text'],
                'score'   : float(scores[idx]),
                'doc_id'  : all_chunks[idx]['doc_id'],
                'pubid'   : all_chunks[idx]['pubid']
            })

    return hits


# Quick test
bm25_results = bm25_search(test_q, bm25_index, all_chunks, top_k=5)
print(f'BM25 search test — top 3 results:')
for i, r in enumerate(bm25_results[:3]):
    print(f'  [{i+1}] score={r["score"]:.4f} | {r["text"][:100]}...')

BM25 search test — top 3 results:
  [1] score=20.9714 | Antiplatelets such as aspirin are widely used to reduce thrombotic events in patients with various c...
  [2] score=17.8478 | Since the 1980s, clinical trial evidence has supported aspirin use in the secondary prevention of ca...
  [3] score=12.7288 | . We found that the LoS of LTCI users is 1.27 days greater than that of non-LTCI users, but the LoS ...


## 4.7 Component 3 — RRF Fusion
Reciprocal Rank Fusion merges semantic + BM25 results into one ranked list.

**Formula:** `RRF_score(d) = Σ 1 / (k + rank(d))`  where k=60 (standard constant)

A chunk that ranks high in BOTH lists gets a very high fused score.

In [ ]:
def reciprocal_rank_fusion(semantic_hits, bm25_hits, k=60, top_k=20):
    """
    Combine semantic and BM25 results using Reciprocal Rank Fusion.

    Args:
        semantic_hits : ranked list from semantic search
        bm25_hits     : ranked list from BM25 search
        k             : RRF constant (60 is standard)
        top_k         : number of results to return

    Returns:
        List of dicts sorted by fused RRF score (descending)
    """
    # Store RRF scores and chunk data
    rrf_scores = defaultdict(float)
    chunk_data = {}

    # Process semantic results
    for rank, hit in enumerate(semantic_hits):
        cid = hit['chunk_id']
        rrf_scores[cid] += 1.0 / (k + rank + 1)
        chunk_data[cid] = hit

    # Process BM25 results
    for rank, hit in enumerate(bm25_hits):
        cid = hit['chunk_id']
        rrf_scores[cid] += 1.0 / (k + rank + 1)
        if cid not in chunk_data:
            chunk_data[cid] = hit

    # Sort by RRF score descending
    sorted_ids = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)

    fused_results = []
    for cid in sorted_ids[:top_k]:
        result = chunk_data[cid].copy()
        result['rrf_score'] = rrf_scores[cid]
        fused_results.append(result)

    return fused_results


# Test RRF
sem_hits  = semantic_search(test_q, bi_encoder, index, top_k=20)
bm25_hits = bm25_search(test_q, bm25_index, all_chunks, top_k=20)
fused     = reciprocal_rank_fusion(sem_hits, bm25_hits, k=60, top_k=10)

print(f'RRF Fusion test — top 3 results:')
for i, r in enumerate(fused[:3]):
    print(f'  [{i+1}] rrf_score={r["rrf_score"]:.5f} | {r["text"][:100]}...')

RRF Fusion test — top 3 results:
  [1] rrf_score=0.03252 | Since the 1980s, clinical trial evidence has supported aspirin use in the secondary prevention of ca...
  [2] rrf_score=0.03105 | . Overall, 547 men (9.5%) were taking aspirin daily, of whom 321 (59%) had documented CVD. Among men...
  [3] rrf_score=0.03089 | Antiplatelets such as aspirin are widely used to reduce thrombotic events in patients with various c...


## 4.8 Component 4 — CrossEncoder Re-Ranking
The CrossEncoder takes each (query, chunk) pair and gives a relevance score.
Much more accurate than bi-encoder but slower — so we only apply it to top 10 fused results.

In [ ]:
def rerank_with_crossencoder(query, fused_results, cross_encoder, top_k=5):
    """
    Re-rank fused results using CrossEncoder.

    Args:
        query          : original user query string
        fused_results  : output of RRF fusion
        cross_encoder  : loaded CrossEncoder model
        top_k          : final number of chunks to return

    Returns:
        Top-k re-ranked chunks with cross_score added
    """
    if not fused_results:
        return []

    # Build (query, passage) pairs
    pairs = [[query, r['text']] for r in fused_results]

    # CrossEncoder scores
    scores = cross_encoder.predict(pairs)

    # Attach scores and sort
    for result, score in zip(fused_results, scores):
        result['cross_score'] = float(score)

    reranked = sorted(fused_results, key=lambda x: x['cross_score'], reverse=True)

    return reranked[:top_k]


# Test re-ranking
reranked = rerank_with_crossencoder(test_q, fused, cross_encoder, top_k=5)

print(f'Re-ranked results — top 3:')
for i, r in enumerate(reranked[:3]):
    print(f'  [{i+1}] cross_score={r["cross_score"]:.4f} | rrf={r["rrf_score"]:.5f}')
    print(f'       {r["text"][:120]}...')

Re-ranked results — top 3:
  [1] cross_score=7.3895 | rrf=0.03089
       Antiplatelets such as aspirin are widely used to reduce thrombotic events in patients with various cardiovascular comorb...
  [2] cross_score=4.5678 | rrf=0.03252
       Since the 1980s, clinical trial evidence has supported aspirin use in the secondary prevention of cardiovascular disease...
  [3] cross_score=0.5861 | rrf=0.02822
       . We calculated risk reductions for combined cardiovascular events, cardiovascular death, stroke, and coronary heart dis...


## 4.9 Full Hybrid Retrieval Pipeline
This is the complete function that combines all 4 components.
This exact function will be used in the final Streamlit app.

In [ ]:
def hybrid_retrieve(
    query,
    bi_encoder,
    cross_encoder,
    pinecone_index,
    bm25_index,
    all_chunks,
    semantic_top_k = 20,
    bm25_top_k     = 20,
    rrf_top_k      = 10,
    final_top_k    = 5,
    rrf_k          = 60
):
    """
    Full hybrid retrieval pipeline:
      1. Semantic search  (Pinecone)
      2. BM25 search      (keyword)
      3. RRF fusion       (combine rankings)
      4. CrossEncoder     (re-rank top results)

    Returns:
        final_top_k best chunks, each with all scores attached
    """
    t0 = time.time()

    # Step 1: Semantic
    sem_hits  = semantic_search(query, bi_encoder, pinecone_index, top_k=semantic_top_k)
    t1 = time.time()

    # Step 2: BM25
    bm25_hits = bm25_search(query, bm25_index, all_chunks, top_k=bm25_top_k)
    t2 = time.time()

    # Step 3: RRF
    fused     = reciprocal_rank_fusion(sem_hits, bm25_hits, k=rrf_k, top_k=rrf_top_k)
    t3 = time.time()

    # Step 4: Re-rank
    final     = rerank_with_crossencoder(query, fused, cross_encoder, top_k=final_top_k)
    t4 = time.time()

    # Timing breakdown (useful for report latency analysis)
    timings = {
        'semantic_ms'  : round((t1 - t0) * 1000, 1),
        'bm25_ms'      : round((t2 - t1) * 1000, 1),
        'rrf_ms'       : round((t3 - t2) * 1000, 1),
        'rerank_ms'    : round((t4 - t3) * 1000, 1),
        'total_ms'     : round((t4 - t0) * 1000, 1)
    }

    return final, timings


# Full pipeline test
print('=== FULL HYBRID RETRIEVAL TEST ===')
print(f'Query: "{test_q}"\n')

results, timings = hybrid_retrieve(
    query          = test_q,
    bi_encoder     = bi_encoder,
    cross_encoder  = cross_encoder,
    pinecone_index = index,
    bm25_index     = bm25_index,
    all_chunks     = all_chunks
)

print('TOP 5 RETRIEVED CHUNKS:')
for i, r in enumerate(results):
    print(f'\n[{i+1}] CrossScore={r["cross_score"]:.4f} | RRF={r["rrf_score"]:.5f}')
    print(f'     PubID : {r["pubid"]}')
    print(f'     Text  : {r["text"][:200]}...')

print(f'\n=== LATENCY BREAKDOWN ===')
for k, v in timings.items():
    print(f'  {k:<15}: {v} ms')

=== FULL HYBRID RETRIEVAL TEST ===
Query: "Does aspirin reduce cardiovascular risk?"

TOP 5 RETRIEVED CHUNKS:

[1] CrossScore=7.3895 | RRF=0.03089
     PubID : 27129549
     Text  : Antiplatelets such as aspirin are widely used to reduce thrombotic events in patients with various cardiovascular comorbidities. Continuing aspirin through noncardiac surgery has been shown to reduce ...

[2] CrossScore=4.5678 | RRF=0.03252
     PubID : 9281867
     Text  : Since the 1980s, clinical trial evidence has supported aspirin use in the secondary prevention of cardiovascular disease (CVD).AIM: To explore aspirin use among British men with known CVD in a populat...

[3] CrossScore=0.5861 | RRF=0.02822
     PubID : 17453746
     Text  : . We calculated risk reductions for combined cardiovascular events, cardiovascular death, stroke, and coronary heart disease from groups of trials using atenolol first-line and all beta-blockers first...

[4] CrossScore=-0.2566 | RRF=0.02963
     PubID : 17432927
   

## 4.10 Test with Multiple Medical Queries
Run 5 diverse queries to make sure the pipeline works well.

In [ ]:
test_queries = [
    'What is the effect of metformin on type 2 diabetes patients?',
    'Does exercise reduce the risk of breast cancer?',
    'What are the side effects of statins on liver function?',
    'Is there a relationship between sleep deprivation and obesity?',
    'How effective is cognitive behavioral therapy for depression?'
]

print('=== MULTI-QUERY TEST ===')
for q in test_queries:
    results, timings = hybrid_retrieve(
        query          = q,
        bi_encoder     = bi_encoder,
        cross_encoder  = cross_encoder,
        pinecone_index = index,
        bm25_index     = bm25_index,
        all_chunks     = all_chunks,
        final_top_k    = 3
    )
    print(f'\nQ: {q}')
    print(f'  Top result ({timings["total_ms"]}ms): {results[0]["text"][:150]}...')
    print(f'  CrossScore: {results[0]["cross_score"]:.4f}')

=== MULTI-QUERY TEST ===

Q: What is the effect of metformin on type 2 diabetes patients?
  Top result (157.8ms): Recent studies suggest that the use of metformin is associated with reduced cancer incidence and improved prognosis in patients with oesophageal cance...
  CrossScore: 2.0595

Q: Does exercise reduce the risk of breast cancer?
  Top result (118.0ms): . On the other hand, compared with non-tea drinkers, women who started tea drinking at 25 years of age or younger had an increased risk of postmenopau...
  CrossScore: -3.1743

Q: What are the side effects of statins on liver function?
  Top result (108.6ms): Statins are among the most prescribed drugs worldwide to reduce the risk of cardiovascular events. Interindividual variability in drug response is a m...
  CrossScore: 2.7592

Q: Is there a relationship between sleep deprivation and obesity?
  Top result (120.0ms): Obstructive sleep apnea syndrome (OSAS) and obesity frequently occur together. The relationship between incre

## 4.11 Save Retrieval Module
Save all functions and config into a Python file for the Streamlit app.

In [ ]:
retrieval_module = '''
# retrieval.py
# Full Hybrid Retrieval Pipeline for PubMedQA RAG System

import time
import numpy as np
from collections import defaultdict


def semantic_search(query, bi_encoder, pinecone_index, top_k=20):
    query_emb = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0].tolist()
    results   = pinecone_index.query(vector=query_emb, top_k=top_k, include_metadata=True)
    hits = []
    for match in results["matches"]:
        hits.append({
            "chunk_id": match["id"],
            "text"    : match["metadata"]["text"],
            "score"   : match["score"],
            "doc_id"  : match["metadata"].get("doc_id", ""),
            "pubid"   : match["metadata"].get("pubid", "")
        })
    return hits


def bm25_search(query, bm25_index, all_chunks, top_k=20):
    tokenized_query = query.lower().split()
    scores          = bm25_index.get_scores(tokenized_query)
    top_indices     = np.argsort(scores)[::-1][:top_k]
    hits = []
    for idx in top_indices:
        if scores[idx] > 0:
            hits.append({
                "chunk_id": all_chunks[idx]["chunk_id"],
                "text"    : all_chunks[idx]["text"],
                "score"   : float(scores[idx]),
                "doc_id"  : all_chunks[idx]["doc_id"],
                "pubid"   : all_chunks[idx]["pubid"]
            })
    return hits


def reciprocal_rank_fusion(semantic_hits, bm25_hits, k=60, top_k=20):
    rrf_scores = defaultdict(float)
    chunk_data = {}
    for rank, hit in enumerate(semantic_hits):
        cid = hit["chunk_id"]
        rrf_scores[cid] += 1.0 / (k + rank + 1)
        chunk_data[cid]  = hit
    for rank, hit in enumerate(bm25_hits):
        cid = hit["chunk_id"]
        rrf_scores[cid] += 1.0 / (k + rank + 1)
        if cid not in chunk_data:
            chunk_data[cid] = hit
    sorted_ids = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    fused = []
    for cid in sorted_ids[:top_k]:
        result = chunk_data[cid].copy()
        result["rrf_score"] = rrf_scores[cid]
        fused.append(result)
    return fused


def rerank_with_crossencoder(query, fused_results, cross_encoder, top_k=5):
    if not fused_results:
        return []
    pairs  = [[query, r["text"]] for r in fused_results]
    scores = cross_encoder.predict(pairs)
    for result, score in zip(fused_results, scores):
        result["cross_score"] = float(score)
    reranked = sorted(fused_results, key=lambda x: x["cross_score"], reverse=True)
    return reranked[:top_k]


def hybrid_retrieve(query, bi_encoder, cross_encoder, pinecone_index,
                    bm25_index, all_chunks, semantic_top_k=20,
                    bm25_top_k=20, rrf_top_k=10, final_top_k=5, rrf_k=60):
    t0        = time.time()
    sem_hits  = semantic_search(query, bi_encoder, pinecone_index, top_k=semantic_top_k)
    t1        = time.time()
    bm25_hits = bm25_search(query, bm25_index, all_chunks, top_k=bm25_top_k)
    t2        = time.time()
    fused     = reciprocal_rank_fusion(sem_hits, bm25_hits, k=rrf_k, top_k=rrf_top_k)
    t3        = time.time()
    final     = rerank_with_crossencoder(query, fused, cross_encoder, top_k=final_top_k)
    t4        = time.time()
    timings   = {
        "semantic_ms": round((t1-t0)*1000, 1),
        "bm25_ms"    : round((t2-t1)*1000, 1),
        "rrf_ms"     : round((t3-t2)*1000, 1),
        "rerank_ms"  : round((t4-t3)*1000, 1),
        "total_ms"   : round((t4-t0)*1000, 1)
    }
    return final, timings
'''

with open('retrieval.py', 'w') as f:
    f.write(retrieval_module)

print('retrieval.py saved ✅')
print('\nStep 4 Complete! ✅')
print('Next: Step 5 — LLM Generation via HuggingFace Inference API')

retrieval.py saved ✅

Step 4 Complete! ✅
Next: Step 5 — LLM Generation via HuggingFace Inference API


In [ ]:
from google.colab import files
files.download('retrieval.py')
print('Downloaded retrieval.py!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded retrieval.py!
